In [7]:
import pandas as pd

In [8]:
df =pd.read_csv('../essay_argument_annotation/processed_asap_aes_data.tsv', delimiter='\t', encoding="ISO-8859-1")
df

,essay_id,essay_set,essay,domain1_score,domain2_score
0,1,1,"Dear local newspaper, I think effects computer...",8,NaN
1,2,1,"Dear @CAPS1 @CAPS2, I believe that using compu...",9,NaN
2,3,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",7,NaN
3,4,1,"Dear Local Newspaper, @CAPS1 I have found that...",10,NaN
4,5,1,"Dear @LOCATION1, I know having computers has a...",8,NaN
...,...,...,...,...,...
10679,16629,6,The one obstacle the builders had when trying ...,0,NaN
10680,16630,6,Some of the problems with the constructing of ...,2,NaN
10681,16631,6,The builders of the Empire State building face...,3,NaN
10682,16632,6,The obstacles the builders of the Empire State...,2,NaN


In [9]:
import os
import re

In [10]:
from segment_essays import load_prompts, create_request

In [11]:
essay_prompts = {}
for prompt_path in os.listdir("../essay_scoring/asap-aes/prompts"):
    idx = int(prompt_path.split('.')[0])
    
    prompts = load_prompts('prompts/argument_segmentation_prompt.txt', os.path.join("../essay_scoring/asap-aes/prompts", prompt_path), 'prompts/argument_evaluation_prompt.txt', )
    essay_prompts[idx] = prompts

In [13]:

for row in df.iterrows():
    row = row[1]
    idx, set_idx, essay = row['essay_id'], row['essay_set'], row['essay']
    # print(essay_prompts[set_idx]['segmentation_prompt'])
    essay = re.sub(r'\s+', " ", essay)
    essay = re.sub(r'\n', r'\\n', essay)
    essay = re.sub(r'\\(?=$|[^n])', "", essay)
    essay = re.sub(r'["’]', "'", essay)
    request = create_request(idx, essay_prompts[set_idx]['segmentation_prompt'], essay)
    # print(request)
    with open('batch_segmentation.jsonl', 'a') as f:
        f.write(request+'\n')

In [14]:
from dotenv import load_dotenv
load_dotenv('.env')

True

In [15]:
from openai import OpenAI
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

batch_input_file = client.files.create(
    file=open("batch_segmentation.jsonl", "rb"),
    purpose="batch"
)

print(batch_input_file)

FileObject(id='file-Evz5NNU2HzCDtKopdsxbPy', bytes=101436449, created_at=1745818406, filename='batch_segmentation.jsonl', object='file', purpose='batch', status='processed', expires_at=None, status_details=None)


In [16]:
batch_input_file_id = batch_input_file.id
client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={
        "description": "batched essay segmentation"
    }
)

Batch(id='batch_680f133173cc81908c6b7cdc195e87f3', completion_window='24h', created_at=1745818417, endpoint='/v1/chat/completions', input_file_id='file-Evz5NNU2HzCDtKopdsxbPy', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1745904817, failed_at=None, finalizing_at=None, in_progress_at=None, metadata={'description': 'batched essay segmentation'}, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0))

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv('.env')
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
client.batches.retrieve('batch_680f133173cc81908c6b7cdc195e87f3')

Batch(id='batch_680f133173cc81908c6b7cdc195e87f3', completion_window='24h', created_at=1745818417, endpoint='/v1/chat/completions', input_file_id='file-Evz5NNU2HzCDtKopdsxbPy', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1745826679, error_file_id=None, errors=None, expired_at=None, expires_at=1745904817, failed_at=None, finalizing_at=1745824985, in_progress_at=1745818430, metadata={'description': 'batched essay segmentation'}, output_file_id='file-FrS5WKXMEYi6CdR65m5Tat', request_counts=BatchRequestCounts(completed=10684, failed=0, total=10684))

In [2]:
file_response = client.files.content("file-FrS5WKXMEYi6CdR65m5Tat")
with open('batched_segmentation_outputs.jsonl', 'w') as f:
    f.write(file_response.text)
# print(file_response.text)

In [32]:
import pandas as pd

In [35]:
df =pd.read_csv('asap_aes_data_segmented.tsv', delimiter='\t', encoding="ISO-8859-1")
df

,essay_id,essay_set,essay,segmented_essays,domain1_score,domain2_score
0,1,1,"Dear local newspaper, I think effects computer...","<Position>Dear local newspaper, I think effect...",8,NaN
1,2,1,"Dear @CAPS1 @CAPS2, I believe that using compu...","<Position>Dear @CAPS1 @CAPS2, I believe that u...",9,NaN
2,3,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...","Dear, @CAPS1 @CAPS2 @CAPS3\n\n<Lead>\nMore and...",7,NaN
3,4,1,"Dear Local Newspaper, @CAPS1 I have found that...","<Lead>Dear Local Newspaper, @CAPS1</Lead>\n<Co...",10,NaN
4,5,1,"Dear @LOCATION1, I know having computers has a...","Dear @LOCATION1,\n\n<Lead>I know having comput...",8,NaN
...,...,...,...,...,...,...
10679,16629,6,The one obstacle the builders had when trying ...,<Position>The one obstacle the builders had wh...,0,NaN
10680,16630,6,Some of the problems with the constructing of ...,<Claim>Some of the problems with the construct...,2,NaN
10681,16631,6,The builders of the Empire State building face...,<Position>The builders of the Empire State bui...,3,NaN
10682,16632,6,The obstacles the builders of the Empire State...,<Position>The obstacles the builders of the Em...,2,NaN


GRE

In [21]:
import pandas as pd
import re
from segment_essays import create_request

In [8]:
df = pd.read_csv("gre-essay-data.tsv", delimiter='\t')

In [9]:
def load_prompt(segmentation_prompt_file, essay_prompt, essay_instructions):
    with open(segmentation_prompt_file, 'r') as f:
        segmentation_prompt = f.read().format(essay_prompt=essay_prompt, essay_instructions=essay_instructions)
    segmentation_prompt = re.sub(r'\s+', " ", segmentation_prompt)
    segmentation_prompt = re.sub(r'\n', r'\\n', segmentation_prompt)
    segmentation_prompt = re.sub(r'\\(?=$|[^n])', "", segmentation_prompt)
    segmentation_prompt = re.sub(r'["’]', "'", segmentation_prompt)
    
    return segmentation_prompt

asap_aes_data_segmented.tsv	    __pycache__
batched_segmentation_outputs.jsonl  README.md
batch_segmentation.jsonl	    requirements.txt
gre-essay-data.tsv		    segment_essays.py
processed_asap_aes_data.tsv	    testing.ipynb
prompts


In [22]:

for row in df.iterrows():
    idx, row = row
    idx, essay_prompt, essay, essay_instructions = idx, row['prompt'], row['essay-text'], row['task-directions']
    prompt = load_prompt("prompts/gre_argument_segmentation_prompt.txt", essay_prompt, essay_instructions)
    essay = re.sub(r'\s+', " ", essay)
    essay = re.sub(r'\n', r'\\n', essay)
    essay = re.sub(r'\\(?=$|[^n])', "", essay)
    essay = re.sub(r'["’]', "'", essay)
    request = create_request(idx, prompt, essay)
    # print(request)
    with open('gre_batch_segmentation.jsonl', 'a') as f:
        f.write(request+'\n')

In [23]:
from dotenv import load_dotenv
load_dotenv('.env')

True

In [25]:
from openai import OpenAI
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

batch_input_file = client.files.create(
    file=open("gre_batch_segmentation.jsonl", "rb"),
    purpose="batch"
)

print(batch_input_file)

FileObject(id='file-6DzKSbncyV6jkiEg5tgBkC', bytes=289255, created_at=1745950369, filename='gre_batch_segmentation.jsonl', object='file', purpose='batch', status='processed', expires_at=None, status_details=None)


In [26]:
batch_input_file_id = batch_input_file.id
client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={
        "description": "batched essay segmentation"
    }
)

Batch(id='batch_681116bd74a881908e6cf3d699d08703', completion_window='24h', created_at=1745950397, endpoint='/v1/chat/completions', input_file_id='file-6DzKSbncyV6jkiEg5tgBkC', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1746036797, failed_at=None, finalizing_at=None, in_progress_at=None, metadata={'description': 'batched essay segmentation'}, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0))

In [27]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv('.env')
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
client.batches.retrieve('batch_681116bd74a881908e6cf3d699d08703')

Batch(id='batch_681116bd74a881908e6cf3d699d08703', completion_window='24h', created_at=1745950397, endpoint='/v1/chat/completions', input_file_id='file-6DzKSbncyV6jkiEg5tgBkC', object='batch', status='in_progress', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1746036797, failed_at=None, finalizing_at=None, in_progress_at=1745950399, metadata={'description': 'batched essay segmentation'}, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=48))

In [ ]:
file_response = client.files.content("file-FrS5WKXMEYi6CdR65m5Tat")
with open('batched_segmentation_outputs.jsonl', 'w') as f:
    f.write(file_response.text)
# print(file_response.text)